# Notebook 03 — Avaliação da geração RAG

**Objetivo:** estabelecer uma linha de base do RAG com Gemini e `top_k=20`, reutilizando as 19 perguntas do Notebook 02. Avaliar separadamente **execução**, **abstenção**, **fidelidade factual**, **qualidade das citações** e **síntese**.

**Importante:** uma citação com ID válido não comprova que ela sustenta a afirmação. A qualidade semântica será avaliada manualmente. O notebook não modifica `src/rag.py` nem o índice.

## 1. Ambiente e parâmetros
Execute o notebook a partir do projeto. A chave fica no `.env`, nunca nas saídas. A execução em lote está **desativada por padrão** para evitar chamadas acidentais à API.

In [ ]:
"""Load project dependencies and establish reproducible evaluation settings."""
from pathlib import Path
import json
import os
import time
from datetime import datetime, timezone

import chromadb
import pandas as pd
from dotenv import load_dotenv
from google import genai
from sentence_transformers import SentenceTransformer

import sys
import hashlib

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src" / "rag.py").exists():
    raise FileNotFoundError(
        f"Não foi possível localizar a raiz do projeto: {PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag import (
    generate_answer, SYSTEM_INSTRUCTION, RESPONSE_SCHEMA, USER_PROMPT_TEMPLATE
)
from src.retrieve import CHROMA_PATH, COLLECTION_NAME, MODEL_NAME

load_dotenv(PROJECT_ROOT / ".env")
MODEL_ID = os.getenv("GEMINI_MODEL")
assert os.getenv("GEMINI_API_KEY"), "Configure GEMINI_API_KEY no .env"
assert MODEL_ID, "Configure GEMINI_MODEL no .env"
TOP_K = 20
PROMPT_COMPONENTS = {
    "system_instruction": SYSTEM_INSTRUCTION,
    "response_schema": RESPONSE_SCHEMA,
    "user_prompt_template": USER_PROMPT_TEMPLATE,
}
PROMPT_SHA256 = hashlib.sha256(
    json.dumps(PROMPT_COMPONENTS, sort_keys=True, ensure_ascii=False).encode("utf-8")
).hexdigest()[:12]
# Set True ONLY if you confirm all legacy runs used the current prompt.
LEGACY_PROMPT_CONFIRMED = False
EVAL_DIR = PROJECT_ROOT / "evaluation"
EVAL_DIR.mkdir(exist_ok=True)
RUNS_PATH = EVAL_DIR / "generation_runs.jsonl"
LABELS_PATH = EVAL_DIR / "generation_labels.csv"
print(f"Modelo: {MODEL_ID} | top_k={TOP_K} | prompt={PROMPT_SHA256} | pasta: {EVAL_DIR}")

## 2. Perguntas de avaliação
Mesmo conjunto e mesma classificação do Notebook 02: nove de domínio, sete fora do escopo e três vagas.

In [ ]:
"""Define in-domain, ambiguous and out-of-scope evaluation questions."""

questions = pd.DataFrame([
    ("Q01", "Entrega", "Quais avaliações relatam que o produto não foi entregue?", "domain"),
    ("Q02", "Entrega", "Quais clientes reclamam de atraso na entrega?", "domain"),
    ("Q03", "Entrega", "Existem relatos de produtos entregues antes do prazo?", "domain"),
    ("Q04", "Qualidade", "Quais avaliações mencionam produtos com defeito?", "domain"),
    ("Q05", "Qualidade", "Quais clientes relatam produtos diferentes do anunciado?", "domain"),
    ("Q06", "Atendimento", "Quais avaliações mencionam dificuldades com o atendimento?", "domain"),
    ("Q07", "Pós-venda", "Existem reclamações sobre devolução ou reembolso?", "domain"),
    ("Q08", "Satisfação", "Quais avaliações elogiam a qualidade dos produtos?", "domain"),
    ("Q09", "Satisfação", "Quais clientes recomendam a experiência de compra?", "domain"),
    ("Q10", "Fora do escopo", "Qual é o melhor smartphone de 2025?", "out_of_scope"),
    ("Q11", "Fora do escopo", "Qual será o faturamento da Olist em 2027?", "out_of_scope"),
    ("Q12", "Fora do escopo", "Qual é a cotação atual do dólar?", "out_of_scope"),
    ("Q13", "Fora do escopo", "Qual é a política de frete da Olist em 2025?", "out_of_scope"),
    ("Q14", "Fora do escopo", "A Olist oferece garantia estendida em 2026?", "out_of_scope"),
    ("Q15", "Fora do escopo", "Quanto custa anunciar na Olist hoje?", "out_of_scope"),
    ("Q16", "Fora do escopo", "Qual o prazo oficial de reembolso da Olist atualmente?", "out_of_scope"),
    ("Q17", "Vaga", "Entrega?", "ambiguous"),
    ("Q18", "Vaga", "Problemas com a compra", "ambiguous"),
    ("Q19", "Vaga", "Foi bom?", "ambiguous"),
], columns=["question_id", "category", "question", "scope"])

assert questions["question_id"].is_unique
display(questions)


## 3. Inicializar os componentes uma única vez
O modelo de embeddings e a coleção são reutilizados em todas as consultas.

In [ ]:
"""Initialize the existing persistent index and Gemini client."""
chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = chroma_client.get_collection(COLLECTION_NAME)
embedding_model = SentenceTransformer(MODEL_NAME)
llm_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("Documentos no índice:", collection.count())

## 4. Executar e registrar uma consulta
Comece com Q01. Cada execução grava um registro JSONL com pergunta, resposta, status, evidências citadas, textos recuperados, modelo e tempo. **Não são armazenadas chaves de API.** Se houver falha técnica, ela permanece `status=erro`, não é contabilizada como abstenção.

In [ ]:
"""Execute one question and persist the complete evidence snapshot."""
def evaluate_one(question_id: str) -> dict:
    """Run RAG once and append an auditable snapshot to JSONL."""
    selected = questions.loc[questions["question_id"] == question_id]
    if len(selected) != 1:
        raise ValueError(f"Pergunta não encontrada: {question_id}")
    item = selected.iloc[0]
    started = time.perf_counter()
    result = generate_answer(
        question=item["question"],
        collection=collection,
        embedding_model=embedding_model,
        llm_client=llm_client,
        llm_model=MODEL_ID,
        top_k=TOP_K,
    )
    record = {
        "question_id": question_id,
        "category": item["category"],
        "scope": item["scope"],
        "question": item["question"],
        "model": MODEL_ID,
        "top_k": TOP_K,
        "prompt_sha256": PROMPT_SHA256,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "elapsed_seconds": round(time.perf_counter() - started, 3),
        "status": result["status"],
        "answer": result["answer"],
        "cited_review_ids": result["cited_review_ids"],
        "insufficient_evidence": result["insufficient_evidence"],
        "insufficiency_reason": result["insufficiency_reason"],
        "limited_evidence": result["limited_evidence"],
        "error": result["error"],
        "retrieval": result["retrieval"],
    }
    with RUNS_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
    return record

# Reuse Q01 only if its configuration and prompt identity are verified.
def matches_experiment(record: dict) -> bool:
    """Match model, retrieval depth and verified prompt identity."""
    if record.get("model") != MODEL_ID or record.get("top_k") != TOP_K:
        return False
    recorded_hash = record.get("prompt_sha256")
    return (recorded_hash == PROMPT_SHA256 or
            (recorded_hash is None and LEGACY_PROMPT_CONFIRMED))

existing_q01 = []
if RUNS_PATH.exists():
    with RUNS_PATH.open(encoding="utf-8") as handle:
        existing_q01 = [
            record for line in handle if line.strip()
            if (record := json.loads(line)).get("question_id") == "Q01"
            and matches_experiment(record)
            and record.get("status") != "erro"
        ]

# Do not call the API automatically if an unverified legacy Q01 exists.
if existing_q01:
    first = existing_q01[-1]
    print("Q01 reutilizada:", first["status"])
elif RUNS_PATH.exists() and any(
    r.get("question_id") == "Q01" and r.get("model") == MODEL_ID
    and r.get("top_k") == TOP_K and r.get("prompt_sha256") is None
    for r in [json.loads(line) for line in RUNS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
):
    first = None
    print("Q01 legada encontrada, mas o prompt não foi verificado. Nenhuma chamada automática.")
else:
    first = None
    print("Q01 não executada automaticamente. Ative o lote conscientemente na seção 6.")

if first is not None:
    print("Status:", first["status"], "| Tempo:", first["elapsed_seconds"], "s")
    print("Resposta:", first["answer"])
    print("Citações:", len(first["cited_review_ids"]))

## 5. Inspeção das fontes citadas
Conferir se cada texto citado **realmente sustenta** o que a resposta afirma. A verificação automática abaixo apenas confere a presença do identificador no contexto.

In [ ]:
"""Display cited review texts and check citation identifier membership."""
def cited_evidence(record: dict) -> pd.DataFrame:
    """Display cited source texts from the stored retrieval snapshot."""
    reviews = record["retrieval"]["reviews"]
    by_id = {review["review_id"]: review for review in reviews}
    return pd.DataFrame([
        {
            "review_id": review_id,
            "id_found": review_id in by_id,
            "document": by_id.get(review_id, {}).get("document", ""),
        }
        for review_id in record["cited_review_ids"]
    ], columns=["review_id", "id_found", "document"])

if first is not None:
    display(cited_evidence(first))
else:
    print("Q01 não carregada: verifique o prompt legado ou execute o lote conscientemente.")

## 6. Lote completo — somente quando você decidir
Mantenha `RUN_BATCH=False` até revisar a primeira resposta. Cada chamada pode consumir cota. A execução de Q01 também reutiliza o resultado já salvo ao reexecutar a seção 4. O código reutiliza apenas execuções bem-sucedidas com a mesma pergunta, modelo, `top_k` e versão do prompt (`prompt_sha256`). Alterações no prompt ou no schema criam uma nova versão do experimento, sem sobrescrever registros anteriores.

**Atenção:** registros antigos não têm `prompt_sha256`. Para reutilizá-los sem novas chamadas, defina `LEGACY_PROMPT_CONFIRMED=True` na seção 1 **somente após confirmar** que as instruções, o schema e o template do prompt de `src/rag.py` não mudaram desde as execuções. Erros técnicos são reexecutáveis quando o lote estiver ativado.


In [ ]:
"""Optionally run only questions missing from the current experiment."""
RUN_BATCH = False

def read_runs(path: Path = RUNS_PATH) -> list[dict]:
    """Read persisted JSONL evaluation records."""
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

if RUN_BATCH:
    existing = {
        r["question_id"]
        for r in read_runs()
        if matches_experiment(r) and r.get("status") != "erro"
    }
    for question_id in questions["question_id"]:
        if question_id in existing:
            print(f"{question_id}: já registrado; ignorado")
            continue
        record = evaluate_one(question_id)
        if record["status"] != "erro":
            existing.add(question_id)
        print(f"{question_id}: {record['status']} ({record['elapsed_seconds']} s)")
else:
    print("Lote desativado. Altere RUN_BATCH para True quando desejar.")

## 7. Consolidar resultados automáticos
Esta tabela descreve comportamento técnico, **não mede fidelidade semântica**. Perguntas fora do escopo têm abstenção esperada; perguntas vagas exigem julgamento contextual e não são automaticamente classificadas como acerto/erro.

In [ ]:
"""Build an execution summary without confusing errors with abstentions."""
runs = read_runs()
latest = {}
for record in runs:
    if matches_experiment(record):
        latest[record["question_id"]] = record
current = [
    r for r in latest.values()
]
summary = pd.DataFrame([{
    "question_id": r["question_id"],
    "scope": r["scope"],
    "status": r["status"],
    "reason": r["insufficiency_reason"],
    "citations": len(r["cited_review_ids"]),
    "retrieved": len(r["retrieval"]["reviews"]),
    "latency_s": r["elapsed_seconds"],
    "out_of_scope_abstention": (
        None if r["status"] == "erro" else r["status"] == "abstencao"
    ) if r["scope"] == "out_of_scope" else None,
} for r in current])
if not summary.empty:
    display(summary.sort_values("question_id"))
    print("Execuções:", len(summary), "| Erros técnicos:", int((summary["status"] == "erro").sum()))
    valid_oos = summary[(summary["scope"] == "out_of_scope") & (summary["status"] != "erro")]
    print("Abstenções fora do escopo:", int((valid_oos["status"] == "abstencao").sum()),
          "/", len(valid_oos), "execuções sem erro")
else:
    print("Nenhuma execução registrada para esta configuração.")

## 8. Planilha de avaliação humana
Para cada pergunta, julgue manualmente:

- **`faithfulness`**: `2` = todas as afirmações sustentadas; `1` = parcialmente sustentadas; `0` = afirmações não sustentadas.
- **`citation_quality`**: `2` = citações suficientes e corretas; `1` = parcialmente corretas/incompletas; `0` = incorretas; deixar vazio se não houver citações.
- **`answer_quality`**: `2` = síntese clara e responde ao pedido; `1` = parcialmente; `0` = inadequada.
- **`abstention_correct`**: `1` = abstenção apropriada; `0` = inapropriada; vazio quando não aplicável.

A pergunta fora do escopo deve ser recusada; uma pergunta de domínio pode ser corretamente recusada se o contexto não trouxer evidências. Não confunda `status=erro` com abstenção.

A célula cria o CSV **apenas se ainda não existir**, preservando rótulos já atribuídos.

Registros com `status=erro` não são criados como novas linhas de avaliação. Rótulos humanos existentes são preservados e o status técnico é sincronizado para a configuração correspondente.


In [ ]:
"""Append new evaluation questions while preserving existing human labels."""
LABEL_COLUMNS = [
    "question_id", "model", "top_k", "prompt_sha256", "status", "faithfulness",
    "citation_quality", "answer_quality", "abstention_correct", "notes",
]
LABEL_KEY = ["question_id", "model", "top_k", "prompt_sha256"]

# Read labels as text so that manual notes and blank ratings are preserved.
if LABELS_PATH.exists():
    labels_existing = pd.read_csv(LABELS_PATH, dtype=str, keep_default_na=False)
    if "prompt_sha256" not in labels_existing.columns:
        labels_existing["prompt_sha256"] = ""  # Legacy rows: prompt unknown.
    missing_columns = set(LABEL_COLUMNS) - set(labels_existing.columns)
    if missing_columns:
        raise ValueError(f"Colunas ausentes no CSV de rótulos: {sorted(missing_columns)}")
    labels_before = labels_existing.copy()
    if LEGACY_PROMPT_CONFIRMED:
        legacy_mask = (
            labels_existing["prompt_sha256"].eq("")
            & labels_existing["model"].eq(MODEL_ID)
            & labels_existing["top_k"].eq(str(TOP_K))
        )
        labels_existing.loc[legacy_mask, "prompt_sha256"] = PROMPT_SHA256
    if labels_existing.duplicated(LABEL_KEY).any():
        raise ValueError("Há perguntas duplicadas para a mesma configuração no CSV de rótulos.")
else:
    labels_existing = pd.DataFrame(columns=LABEL_COLUMNS)
    labels_before = labels_existing.copy()

# Sync only technical status, never overwrite human judgments.
status_by_key = {
    (r["question_id"], r["model"], str(r["top_k"]),
     r.get("prompt_sha256") or (PROMPT_SHA256 if LEGACY_PROMPT_CONFIRMED else "")): r["status"]
    for r in current
}
labels_existing["status"] = [
    status_by_key.get((q, m, str(k), h),
        status_by_key.get((q, m, str(k), PROMPT_SHA256), s)
        if h == "" and LEGACY_PROMPT_CONFIRMED else s)
    for q, m, k, h, s in labels_existing[
        ["question_id", "model", "top_k", "prompt_sha256", "status"]
    ].itertuples(index=False, name=None)
]
existing_keys = {
    tuple(str(value) for value in row)
    for row in labels_existing[LABEL_KEY].itertuples(index=False, name=None)
}
new_rows = []
for record in sorted(current, key=lambda item: item["question_id"]):
    if record["status"] == "erro":
        continue
    prompt_hash = record.get("prompt_sha256") or (PROMPT_SHA256 if LEGACY_PROMPT_CONFIRMED else "")
    key = (record["question_id"], record["model"], str(record["top_k"]), prompt_hash)
    if key in existing_keys:
        continue
    new_rows.append({
        "question_id": record["question_id"],
        "model": record["model"],
        "top_k": str(record["top_k"]),
        "prompt_sha256": prompt_hash,
        "status": record["status"],
        "faithfulness": "",
        "citation_quality": "",
        "answer_quality": "",
        "abstention_correct": "",
        "notes": "",
    })
    existing_keys.add(key)

changed = not labels_existing.equals(labels_before)
if new_rows or changed or not LABELS_PATH.exists():
    updated = pd.concat(
        [labels_existing, pd.DataFrame(new_rows, columns=LABEL_COLUMNS)],
        ignore_index=True,
    )
    updated.to_csv(LABELS_PATH, index=False)
    print(f"Planilha atualizada: {len(new_rows)} pergunta(s) adicionada(s); status sincronizado.")
else:
    print("Planilha preservada: nenhuma pergunta nova a adicionar.")

labels_preview = pd.read_csv(LABELS_PATH, dtype=str, keep_default_na=False)
display(labels_preview.head())
print("Perguntas com resposta ou abstenção para avaliar:",
      int(labels_preview["status"].isin(["respondida", "abstencao"]).sum()))

## 9. Resumo dos julgamentos humanos
Preencha `evaluation/generation_labels.csv` fora do notebook e reexecute a célula. Só são exibidas médias dos valores efetivamente preenchidos; campos vazios não são tratados como zero. **Não interpretar médias parciais como avaliação final das 19 perguntas.**

In [ ]:
"""Summarize only manually completed ratings, without inventing scores."""
labels = pd.read_csv(LABELS_PATH, dtype={"prompt_sha256": str})
for col in ["faithfulness", "citation_quality", "answer_quality", "abstention_correct"]:
    labels[col] = pd.to_numeric(labels[col], errors="coerce")
    valid = {0, 1} if col == "abstention_correct" else {0, 1, 2}
    assert labels[col].dropna().isin(valid).all(), f"Rótulo inválido em {col}"
ratings = pd.DataFrame([
    {"metric": col, "labeled": int(labels[col].notna().sum()),
     "mean": labels[col].mean()}
    for col in ["faithfulness", "citation_quality", "answer_quality", "abstention_correct"]
])
display(ratings)
completed = sum(r["status"] != "erro" for r in current)
judgable = int(labels["status"].isin(["respondida", "abstencao"]).sum())
print(f"Perguntas com execução válida: {completed}/19 | Linhas avaliáveis: {judgable}/19")

## 10. Conclusões (preencher após a avaliação)

Registre aqui os resultados observados, os casos de alucinação ou citações inadequadas, os motivos de abstenção e a decisão sobre manter ou revisar o prompt. **Não afirmar que o RAG é confiável apenas porque os testes unitários passaram.**

**Próximo experimento, se necessário:** comparar `top_k=20` com `top_k=30` mantendo modelo, perguntas e critérios de julgamento fixos.